# 19 — Bayesian Optimization / 贝叶斯优化

**Chapter 19 — File 3 of 4**

## Summary / 汇总

This notebook implements the complete Bayesian optimization loop from scratch. It combines the surrogate model with a Probability of Improvement (PoI) acquisition function to iteratively select points.

本笔记本从头实现完整的贝叶斯优化循环。它将代理模型与概率改进(PoI)采集函数结合起来以迭代地选择点。

## Step 1 — Define Acquisition Function / 定义采集函数

In [ ]:
from scipy.stats import norm# probability of improvement acquisition functiondef acquisition(X, Xsamples, model):    # calculate the best surrogate score found so far    yhat, _ = surrogate(model, X)    best = max(yhat)    # calculate mean and stdev via surrogate function    mu, std = surrogate(model, Xsamples)    mu = mu[:, 0]    # calculate the probability of improvement    # This measures how likely each point is to improve upon the best found    probs = norm.cdf((mu - best) / (std+1E-9))    return probs

## Step 2 — Define Acquisition Optimization / 定义采集优化

In [ ]:
# optimize the acquisition functiondef opt_acquisition(X, y, model):    # random search: generate random samples in the domain    Xsamples = random(100)    Xsamples = Xsamples.reshape(len(Xsamples), 1)    # calculate the acquisition function for each sample    scores = acquisition(X, Xsamples, model)    # locate the index of the largest scores (best acquisition value)    ix = argmax(scores)    # Return the point with highest acquisition value    return Xsamples[ix, 0]

## Step 3 — Main Bayesian Optimization Loop / 主贝叶斯优化循环

In [ ]:
from numpy import vstack# sample the domain sparsely with noiseX = random(100)y = asarray([objective(x) for x in X])# reshape into rows and colsX = X.reshape(len(X), 1)y = y.reshape(len(y), 1)# define the modelmodel = GaussianProcessRegressor()# fit the modelmodel.fit(X, y)# perform the optimization processfor i in range(100):    # select the next point to sample using acquisition function    x = opt_acquisition(X, y, model)    # sample the point (expensive objective function call)    actual = objective(x)    # summarize the finding    est, _ = surrogate(model, [[x]])    print('>x=%.3f, f()=%3f, actual=%.3f' % (x, est, actual))    # add the data to the dataset    X = vstack((X, [[x]]))    y = vstack((y, [[actual]]))    # update the model with new data    model.fit(X, y)# plot all samples and the final surrogate functionplot(X, y, model)# best resultix = argmax(y)print('Best Result: x=%.3f, y=%.3f' % (X[ix], y[ix]))

## Learning Notes / 学习笔记

- **Concept**: Probability of Improvement balances exploration (high uncertainty) and exploitation (high predicted value). It quantifies the likelihood that a point will improve upon the current best observation. **概念**: 改进概率平衡探索（高不确定性）和利用（高预测值）。它量化了一个点改进当前最佳观测的可能性。

- **ML Application**: This loop iteratively refines the surrogate model while efficiently exploring the function space. By 100 iterations, the algorithm typically finds near-optimal solutions with far fewer function evaluations than grid search. **机器学习应用**: 此循环在有效探索函数空间的同时迭代地优化代理模型。经过100次迭代后，该算法通常比网格搜索少调用许多次函数即可找到接近最优的解。

➡️ **Next**: `04_bayes_opt_hyperparam.ipynb`

## Complete Code / 完整代码一览

In [ ]:
# example of bayesian optimization for a 1d function from scratchfrom math import sinfrom math import pifrom numpy import arangefrom numpy import vstackfrom numpy import argmaxfrom numpy import asarrayfrom numpy.random import normalfrom numpy.random import randomfrom scipy.stats import normfrom sklearn.gaussian_process import GaussianProcessRegressorfrom warnings import catch_warningsfrom warnings import simplefilterfrom matplotlib import pyplot# objective functiondef objective(x, noise=0.1):    noise = normal(loc=0, scale=noise)    return (x**2 * sin(5 * pi * x)**6.0) + noise# surrogate or approximation for the objective functiondef surrogate(model, X):    # catch any warning generated when making a prediction    with catch_warnings():        # ignore generated warnings        simplefilter("ignore")        return model.predict(X, return_std=True)# probability of improvement acquisition functiondef acquisition(X, Xsamples, model):    # calculate the best surrogate score found so far    yhat, _ = surrogate(model, X)    best = max(yhat)    # calculate mean and stdev via surrogate function    mu, std = surrogate(model, Xsamples)    mu = mu[:, 0]    # calculate the probability of improvement    probs = norm.cdf((mu - best) / (std+1E-9))    return probs# optimize the acquisition functiondef opt_acquisition(X, y, model):    # random search, generate random samples    Xsamples = random(100)    Xsamples = Xsamples.reshape(len(Xsamples), 1)    # calculate the acquisition function for each sample    scores = acquisition(X, Xsamples, model)    # locate the index of the largest scores    ix = argmax(scores)    return Xsamples[ix, 0]# plot real observations vs surrogate functiondef plot(X, y, model):    # scatter plot of inputs and real objective function    pyplot.scatter(X, y)    # line plot of surrogate function across domain    Xsamples = asarray(arange(0, 1, 0.001))    Xsamples = Xsamples.reshape(len(Xsamples), 1)    ysamples, _ = surrogate(model, Xsamples)    pyplot.plot(Xsamples, ysamples)    # show the plot    pyplot.show()# sample the domain sparsely with noiseX = random(100)y = asarray([objective(x) for x in X])# reshape into rows and colsX = X.reshape(len(X), 1)y = y.reshape(len(y), 1)# define the modelmodel = GaussianProcessRegressor()# fit the modelmodel.fit(X, y)# plot before handplot(X, y, model)# perform the optimization processfor i in range(100):    # select the next point to sample    x = opt_acquisition(X, y, model)    # sample the point    actual = objective(x)    # summarize the finding    est, _ = surrogate(model, [[x]])    print('>x=%.3f, f()=%3f, actual=%.3f' % (x, est, actual))    # add the data to the dataset    X = vstack((X, [[x]]))    y = vstack((y, [[actual]]))    # update the model    model.fit(X, y)# plot all samples and the final surrogate functionplot(X, y, model)# best resultix = argmax(y)print('Best Result: x=%.3f, y=%.3f' % (X[ix], y[ix]))